# 🛒 Superstore Sales — Exploratory Data Analysis
> **Author:** Your Name | **Date:** 2024 | **Tools:** Python, Pandas, Matplotlib, Seaborn

---
## 🎯 Objective
Analyze a retail superstore's sales data to uncover business insights:
- Which categories and sub-categories drive the most revenue?
- Which regions are performing best?
- What are the monthly/seasonal sales trends?
- Which customer segments are most profitable?
- What discounts are hurting or helping profit?

In [ ]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5), 'font.size': 11})

print('✅ Libraries loaded successfully!')

In [ ]:
# ─── 2. LOAD DATA ─────────────────────────────────────────────────────────────
# Download dataset from: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
# Or generate sample data below:

import random
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

n = 1000
categories = {'Furniture': ['Chairs','Tables','Bookcases','Furnishings'],
              'Technology': ['Phones','Accessories','Machines','Copiers'],
              'Office Supplies': ['Binders','Paper','Storage','Appliances','Labels','Fasteners']}
regions = ['West','East','Central','South']
segments = ['Consumer','Corporate','Home Office']
ship_modes = ['Standard Class','Second Class','First Class','Same Day']
states = {'West':['California','Washington','Oregon'],'East':['New York','Pennsylvania','Ohio'],
          'Central':['Texas','Illinois','Michigan'],'South':['Florida','North Carolina','Georgia']}

rows = []
base_date = datetime(2021, 1, 1)
for i in range(n):
    region = random.choice(regions)
    cat = random.choice(list(categories.keys()))
    sub = random.choice(categories[cat])
    sales = round(np.random.lognormal(5, 1), 2)
    discount = round(random.choice([0, 0, 0, 0.1, 0.2, 0.3, 0.4, 0.5]), 2)
    profit = round(sales * np.random.uniform(-0.1, 0.4) * (1 - discount * 1.5), 2)
    qty = random.randint(1, 10)
    order_date = base_date + timedelta(days=random.randint(0, 730))
    rows.append({
        'Order ID': f'CA-{2021+i//500}-{1000+i}',
        'Order Date': order_date,
        'Ship Mode': random.choice(ship_modes),
        'Segment': random.choice(segments),
        'Region': region,
        'State': random.choice(states[region]),
        'Category': cat,
        'Sub-Category': sub,
        'Sales': sales,
        'Quantity': qty,
        'Discount': discount,
        'Profit': profit
    })

df = pd.DataFrame(rows)
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Month'] = df['Order Date'].dt.to_period('M')
df['Year'] = df['Order Date'].dt.year
df['Month_Name'] = df['Order Date'].dt.strftime('%b')

print(f'✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ─── 3. DATA OVERVIEW ─────────────────────────────────────────────────────────
print('=' * 50)
print('📋 DATASET OVERVIEW')
print('=' * 50)
print(f'Shape        : {df.shape}')
print(f'Date Range   : {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Total Sales  : ${df["Sales"].sum():,.2f}')
print(f'Total Profit : ${df["Profit"].sum():,.2f}')
print(f'Profit Margin: {df["Profit"].sum()/df["Sales"].sum()*100:.1f}%')
print()
print('🔍 Missing Values:')
print(df.isnull().sum())
print()
print('📊 Data Types:')
print(df.dtypes)

In [ ]:
# ─── 4. SALES BY CATEGORY ─────────────────────────────────────────────────────
cat_sales = df.groupby('Category')[['Sales','Profit']].sum().sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart
cat_sales['Sales'].plot(kind='bar', ax=axes[0], color=['#2196F3','#FF9800','#4CAF50'], edgecolor='white', linewidth=1.5)
axes[0].set_title('💰 Total Sales by Category', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].annotate(f'${bar.get_height():,.0f}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                     ha='center', va='bottom', fontsize=10, fontweight='bold')

# Profit comparison
x = np.arange(len(cat_sales))
width = 0.35
axes[1].bar(x - width/2, cat_sales['Sales'], width, label='Sales', color='#2196F3', alpha=0.8)
axes[1].bar(x + width/2, cat_sales['Profit'], width, label='Profit', color='#4CAF50', alpha=0.8)
axes[1].set_title('📊 Sales vs Profit by Category', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xticks(x)
axes[1].set_xticklabels(cat_sales.index)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].legend()

plt.tight_layout()
plt.savefig('charts/01_sales_by_category.png', bbox_inches='tight')
plt.show()

print('\n📌 KEY INSIGHT: Technology leads in sales but Office Supplies may have better margins.')

In [ ]:
# ─── 5. REGIONAL PERFORMANCE ─────────────────────────────────────────────────
import os
os.makedirs('charts', exist_ok=True)

region_data = df.groupby('Region')[['Sales','Profit']].sum().reset_index()
region_data['Margin%'] = (region_data['Profit'] / region_data['Sales'] * 100).round(1)
region_data = region_data.sort_values('Sales', ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#E91E63','#2196F3','#FF9800','#4CAF50']

# Sales pie
axes[0].pie(region_data['Sales'], labels=region_data['Region'], autopct='%1.1f%%',
            colors=colors, startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('🌎 Sales Share by Region', fontsize=13, fontweight='bold')

# Profit bar
bars = axes[1].barh(region_data['Region'], region_data['Profit'], color=colors)
axes[1].set_title('💹 Profit by Region', fontsize=13, fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for bar, val in zip(bars, region_data['Profit']):
    axes[1].text(val + 50, bar.get_y() + bar.get_height()/2, f'${val:,.0f}', va='center', fontsize=9)

# Margin bar
bar_colors = ['#4CAF50' if m > 10 else '#FF9800' if m > 0 else '#F44336' for m in region_data['Margin%']]
bars2 = axes[2].bar(region_data['Region'], region_data['Margin%'], color=bar_colors, edgecolor='white')
axes[2].set_title('📈 Profit Margin % by Region', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Margin %')
axes[2].axhline(y=region_data['Margin%'].mean(), color='red', linestyle='--', alpha=0.7, label='Avg')
axes[2].legend()
for bar, val in zip(bars2, region_data['Margin%']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/02_regional_performance.png', bbox_inches='tight')
plt.show()

print(region_data.to_string(index=False))

In [ ]:
# ─── 6. MONTHLY SALES TREND ───────────────────────────────────────────────────
monthly = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum().reset_index()
monthly['Order Date'] = monthly['Order Date'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(monthly)), monthly['Sales'], alpha=0.15, color='#2196F3')
ax.plot(range(len(monthly)), monthly['Sales'], color='#2196F3', linewidth=2.5, marker='o', markersize=4)

# Highlight max/min
max_idx = monthly['Sales'].idxmax()
min_idx = monthly['Sales'].idxmin()
ax.scatter([max_idx], [monthly['Sales'][max_idx]], color='#4CAF50', s=150, zorder=5, label=f'Peak: ${monthly["Sales"][max_idx]:,.0f}')
ax.scatter([min_idx], [monthly['Sales'][min_idx]], color='#F44336', s=150, zorder=5, label=f'Low: ${monthly["Sales"][min_idx]:,.0f}')

ax.set_title('📅 Monthly Sales Trend (2021–2022)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['Order Date'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
ax.set_ylabel('Sales ($)')

plt.tight_layout()
plt.savefig('charts/03_monthly_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── 7. DISCOUNT vs PROFIT ANALYSIS ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Discount vs Profit
scatter = axes[0].scatter(df['Discount'], df['Profit'], alpha=0.4, 
                           c=df['Profit'], cmap='RdYlGn', s=30)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.7)
axes[0].axvline(x=0.2, color='orange', linestyle='--', alpha=0.7, label='20% threshold')
axes[0].set_title('🔻 Discount vs Profit', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Discount Rate')
axes[0].set_ylabel('Profit ($)')
axes[0].legend()
plt.colorbar(scatter, ax=axes[0], label='Profit')

# Avg profit by discount band
df['Discount Band'] = pd.cut(df['Discount'], bins=[-0.01,0,0.1,0.2,0.3,0.5],
                              labels=['0%','1-10%','11-20%','21-30%','31-50%'])
disc_profit = df.groupby('Discount Band')['Profit'].mean()
colors_d = ['#4CAF50' if v > 0 else '#F44336' for v in disc_profit]
disc_profit.plot(kind='bar', ax=axes[1], color=colors_d, edgecolor='white')
axes[1].axhline(y=0, color='black', linewidth=0.8)
axes[1].set_title('📉 Avg Profit by Discount Band', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Discount Band')
axes[1].set_ylabel('Avg Profit ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('charts/04_discount_profit.png', bbox_inches='tight')
plt.show()

print('\n📌 KEY INSIGHT: Discounts above 20% consistently result in negative profit!')

In [ ]:
# ─── 8. TOP 10 SUB-CATEGORIES ─────────────────────────────────────────────────
sub_perf = df.groupby('Sub-Category')[['Sales','Profit']].sum().sort_values('Sales', ascending=True).tail(10)

fig, ax = plt.subplots(figsize=(12, 6))
bar_colors = ['#4CAF50' if p > 0 else '#F44336' for p in sub_perf['Profit']]
bars = ax.barh(sub_perf.index, sub_perf['Sales'], color='#2196F3', alpha=0.7, label='Sales')
ax.barh(sub_perf.index, sub_perf['Profit'], color=bar_colors, alpha=0.9, label='Profit')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('🏆 Top 10 Sub-Categories: Sales vs Profit', fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Amount ($)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('charts/05_subcategory_performance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ─── 9. CUSTOMER SEGMENT ANALYSIS ─────────────────────────────────────────────
seg_data = df.groupby('Segment')[['Sales','Profit']].sum()
seg_data['Margin%'] = (seg_data['Profit']/seg_data['Sales']*100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
seg_colors = ['#9C27B0','#2196F3','#FF9800']

seg_data['Sales'].plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=seg_colors,
                        startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('👥 Sales by Customer Segment', fontsize=13, fontweight='bold')
axes[0].set_ylabel('')

x = np.arange(3)
axes[1].bar(x - 0.2, seg_data['Sales'], 0.35, label='Sales', color=seg_colors, alpha=0.7)
ax2 = axes[1].twinx()
ax2.plot(x, seg_data['Margin%'], 'D-', color='black', markersize=8, linewidth=2, label='Margin %')
axes[1].set_title('💼 Segment Sales & Margin', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(seg_data.index)
axes[1].set_ylabel('Sales ($)')
ax2.set_ylabel('Margin %')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('charts/06_segment_analysis.png', bbox_inches='tight')
plt.show()

print(seg_data)

## 📋 Summary of Key Findings

| # | Insight | Action |
|---|---------|--------|
| 1 | **Technology** generates the highest revenue | Focus marketing on upselling tech products |
| 2 | **West region** leads in both sales and profit | Replicate West's strategy in underperforming regions |
| 3 | **Discounts > 20%** lead to negative profit | Cap discounts at 20% company-wide |
| 4 | **Consumer segment** is the largest revenue driver | Invest in loyalty programs for consumers |
| 5 | Sales peak in **Q4** (holiday season) | Plan inventory and campaigns ahead of Q4 |

---
*Analysis by [Your Name] | Data Source: Superstore Dataset*